In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
cd /content/drive/MyDrive/247/emg2qwerty_FinalProject-main

/content/drive/MyDrive/247/emg2qwerty_FinalProject-main


In [2]:
!pip install -r requirements.txt

  Using cached https://github.com/kpu/kenlm/archive/master.zip
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import glob
from pathlib import Path
DATA_DIR = Path('Data')

all_sessions = sorted(DATA_DIR.rglob('*.hdf5'))
if not all_sessions:
    raise FileNotFoundError(f'No HDF5 files found under {DATA_DIR}')
print(f'Found {len(all_sessions)} session(s):', [s.name for s in all_sessions])

n = len(all_sessions)
train_sessions = all_sessions[: int(n * 0.70)]
val_sessions   = all_sessions[int(n * 0.70) : int(n * 0.85)]
test_sessions  = all_sessions[int(n * 0.85) :]

if not train_sessions: train_sessions = all_sessions
if not val_sessions:   val_sessions   = all_sessions
if not test_sessions:  test_sessions  = all_sessions

WINDOW_LENGTH = 600
PADDING = (150, 150)
BATCH_SIZE = 32
NUM_WORKERS = 2
# model
IN_FEATURES = 528
MLP_FEATURES = [384, 384]
HIDDEN_SIZE = 256
NUM_LAYERS = 3
RNN_TYPE = 'LSTM'
DROPOUT = 0.2
BIDIRECTIONAL = True
#training
MAX_EPOCHS = 50
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5

Found 18 session(s): ['2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-02-1622682789-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-03-1622764398-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-03-1622766673-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-04-1622861066-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-04-1622862148-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-05-1622884635-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b

In [4]:
from collections.abc import Sequence
from typing import Any, ClassVar

import numpy as np
import pytorch_lightning as pl
import torch
from torch import nn
from torchmetrics import MetricCollection

from emg2qwerty.charset import charset
from emg2qwerty.data import LabelData
from emg2qwerty.metrics import CharacterErrorRates
from emg2qwerty.modules import (
    MultiBandRotationInvariantMLP,
    RNNEncoder,
    SpectrogramNorm,
)


class RNNCTCModule(pl.LightningModule):

    NUM_BANDS:         ClassVar[int] = 2
    ELECTRODE_CHANNELS: ClassVar[int] = 16

    def __init__(
        self,
        in_features:   int,
        mlp_features:  Sequence[int],
        hidden_size:   int   = 256,
        num_layers:    int   = 3,
        rnn_type:      str   = 'LSTM',
        dropout:       float = 0.2,
        bidirectional: bool  = True,
        lr:            float = 5e-4,
        weight_decay:  float = 1e-5,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        num_features = self.NUM_BANDS * mlp_features[-1]

        self.model = nn.Sequential(
            SpectrogramNorm(channels=self.NUM_BANDS * self.ELECTRODE_CHANNELS),
            MultiBandRotationInvariantMLP(
                in_features=in_features,
                mlp_features=mlp_features,
                num_bands=self.NUM_BANDS,
            ),
            nn.Flatten(start_dim=2),
            RNNEncoder(
                num_features=num_features,
                hidden_size=hidden_size,
                num_layers=num_layers,
                rnn_type=rnn_type,
                dropout=dropout,
                bidirectional=bidirectional,
            ),
            nn.Linear(num_features, charset().num_classes),
            nn.LogSoftmax(dim=-1),
        )

        self.ctc_loss = nn.CTCLoss(blank=charset().null_class)

        metrics = MetricCollection([CharacterErrorRates()])
        self.metrics = nn.ModuleDict({
            f'{phase}_metrics': metrics.clone(prefix=f'{phase}/')
            for phase in ['train', 'val', 'test']
        })
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.model(inputs)
    def _step(self, phase: str, batch: dict) -> torch.Tensor:
        inputs         = batch['inputs']
        targets        = batch['targets']
        input_lengths  = batch['input_lengths']
        target_lengths = batch['target_lengths']
        N = len(input_lengths)

        emissions = self.forward(inputs)
        T_diff = inputs.shape[0] - emissions.shape[0]
        emission_lengths = input_lengths - T_diff

        loss = self.ctc_loss(
            log_probs=emissions,
            targets=targets.transpose(0, 1),
            input_lengths=emission_lengths,
            target_lengths=target_lengths,
        )
        greedy = emissions.argmax(dim=-1)  # (T, N)
        metrics = self.metrics[f'{phase}_metrics']
        targets_np        = targets.detach().cpu().numpy()
        target_lengths_np = target_lengths.detach().cpu().numpy()
        greedy_np         = greedy.detach().cpu().numpy()

        for i in range(N):
            seq = greedy_np[:, i].tolist()
            decoded = []
            prev = None
            for c in seq:
                if c != charset().null_class and c != prev:
                    decoded.append(c)
                prev = c
            pred   = LabelData.from_labels(decoded)
            target = LabelData.from_labels(targets_np[: target_lengths_np[i], i])
            metrics.update(prediction=pred, target=target)

        self.log(f'{phase}/loss', loss, batch_size=N, sync_dist=True, prog_bar=True)
        return loss

    def _epoch_end(self, phase: str) -> None:
        metrics = self.metrics[f'{phase}_metrics']
        self.log_dict(metrics.compute(), sync_dist=True)
        metrics.reset()

    def training_step(self,   batch, batch_idx): return self._step('train', batch)
    def validation_step(self, batch, batch_idx): return self._step('val',   batch)
    def test_step(self,       batch, batch_idx): return self._step('test',  batch)

    def on_train_epoch_end(self):      self._epoch_end('train')
    def on_validation_epoch_end(self): self._epoch_end('val')
    def on_test_epoch_end(self):       self._epoch_end('test')
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
        )
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}


model = RNNCTCModule(
    in_features=IN_FEATURES,
    mlp_features=MLP_FEATURES,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    rnn_type=RNN_TYPE,
    dropout=DROPOUT,
    bidirectional=BIDIRECTIONAL,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready — {total_params:,} trainable parameters')

Model ready — 6,428,835 trainable parameters


In [5]:
class BandpassFilter:
    """Keep only the 20-500 Hz range where EMG signal lives."""
    def __init__(self, low=20, high=500, fs=2000):
        from scipy.signal import butter, filtfilt
        self.b, self.a = butter(4, [low, high], btype='band', fs=fs)
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        import numpy as np
        from scipy.signal import filtfilt
        x_np = x.numpy()
        x_filtered = filtfilt(self.b, self.a, x_np, axis=0).copy()
        return torch.from_numpy(x_filtered.astype(np.float32))

class NotchFilter:
    """Remove 60 Hz power line noise."""
    def __init__(self, freq=60, fs=2000):
        from scipy.signal import iirnotch
        self.b, self.a = iirnotch(freq, Q=30, fs=fs)
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        import numpy as np
        from scipy.signal import filtfilt
        x_np = x.numpy()
        x_filtered = filtfilt(self.b, self.a, x_np, axis=0).copy()
        return torch.from_numpy(x_filtered.astype(np.float32))

In [7]:
from torch.utils.data import ConcatDataset, DataLoader
from emg2qwerty.data import WindowedEMGDataset
from emg2qwerty.transforms import Compose, ToTensor, LogSpectrogram, RandomBandRotation, SpecAugment

transform = Compose([ToTensor(), LogSpectrogram()])
train_transform = Compose([ToTensor(), NotchFilter(), BandpassFilter(), LogSpectrogram(), RandomBandRotation(), SpecAugment(),])
def make_dataset(sessions, window_length, padding, jitter, transformation):
    return ConcatDataset([
        WindowedEMGDataset(
            p,
            transform=transformation,
            window_length=window_length,
            padding=padding,
            jitter=jitter,
        )
        for p in sessions
    ])
zeropad = (0,0)
train_ds = make_dataset(train_sessions, WINDOW_LENGTH, PADDING, True, train_transform)
val_ds   = make_dataset(val_sessions,   WINDOW_LENGTH, PADDING, False, transform)
test_ds  = make_dataset(test_sessions,  None , zeropad , False , transform)

collate = WindowedEMGDataset.collate

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,  shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,  shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1,           shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)

print(f'Train windows : {len(train_ds):,}')
print(f'Val   windows : {len(val_ds):,}')
print(f'Test  sessions: {len(test_ds):,}')

Train windows : 40,021
Val   windows : 8,866
Test  sessions: 3


In [8]:
import inspect
import emg2qwerty.transforms as T
print([name for name in dir(T) if not name.startswith('_')])

['Any', 'Callable', 'Compose', 'ForEach', 'Lambda', 'LogSpectrogram', 'RandomBandRotation', 'Sequence', 'SpecAugment', 'TTransformIn', 'TTransformOut', 'TemporalAlignmentJitter', 'ToTensor', 'Transform', 'TypeVar', 'dataclass', 'np', 'torch', 'torchaudio']


In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_cb = ModelCheckpoint(
    dirpath='checkpoints/rnn',
    filename='epoch={epoch:02d}-val_cer={val/cer:.4f}',
    monitor='val/CER',
    mode='min',
    save_top_k=3,
    auto_insert_metric_name=False,
)

early_stop_cb = EarlyStopping(
    monitor='val/CER',
    patience=10,
    mode='min',
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='cpu',
    devices=1,
    callbacks=[checkpoint_cb, early_stop_cb],
    log_every_n_steps=10,
    gradient_clip_val=1.0,
)

trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:604: UserWarning: Checkpoint directory checkpoints/rnn exists and is not empty.
  rank_zero_warn(f"Checkpoint directory {dirpath} exists and is not empty.")
INFO:pytorch_lightning.callbacks.model_summary:
  | Name     | Type       | Params
----------------------------------------
0 | model    | Sequential | 6.4 M 
1 | ctc_loss | CTCLoss    | 0     
2 | metrics  | ModuleDict | 0     
----------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Total params
25.715    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

In [ ]:
trainer.test(model, test_loader, ckpt_path='best')

INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at checkpoints/rnn/epoch=12-val_cer=0.0000.ckpt
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from checkpoint at checkpoints/rnn/epoch=12-val_cer=0.0000.ckpt


Testing: 0it [00:00, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   Runningstage.testing    ┃                           ┃
┃          metric           ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/CER          │    27.510751724243164     │
│         test/DER          │    1.7203651666641235     │
│         test/IER          │     6.662642478942871     │
│         test/SER          │    19.127744674682617     │
│         test/loss         │    0.8768424391746521     │
└───────────────────────────┴───────────────────────────┘

[{'test/loss': 0.8768424391746521,
  'test/CER': 27.510751724243164,
  'test/IER': 6.662642478942871,
  'test/DER': 1.7203651666641235,
  'test/SER': 19.127744674682617}]

In [ ]:
model.eval()
device = next(model.parameters()).device

batch = next(iter(val_loader))
inputs         = batch['inputs'].to(device)
targets        = batch['targets']
target_lengths = batch['target_lengths']

with torch.no_grad():
    emissions = model(inputs)

greedy = emissions.argmax(dim=-1).cpu().numpy()
targets_np = targets.numpy()
target_lengths_np = target_lengths.numpy()

null_class = charset().null_class

for i in range(min(4, inputs.shape[1])):
    seq = greedy[:, i].tolist()
    decoded_indices = []
    prev = None
    for c in seq:
        if c != null_class and c != prev:
            decoded_indices.append(c)
        prev = c

    pred_chars   = ''.join(charset().index_to_char(c) for c in decoded_indices)
    target_chars = ''.join(
        charset().index_to_char(int(targets_np[t, i]))
        for t in range(target_lengths_np[i])
    )
    print(f'[{i}] target : {target_chars!r}')
    print(f'[{i}] predict: {pred_chars!r}')
    print()

[0] target : ''
[0] predict: ''

[1] target : ''
[1] predict: ''

[2] target : ''
[2] predict: ''

[3] target : ''
[3] predict: ''



In [ ]:
print(dir(charset()))

['CHAR_SUBSTITUTIONS', 'CHAR_TO_UNICODE', 'KEY_TO_UNICODE', 'MODIFIER_TO_UNICODE', 'UNICHAR_TO_KEY', '__annotations__', '__class__', '__contains__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__post_init__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_key_to_unicode', '_normalize_keys', '_normalize_str', '_unicode_to_key', 'allowed_chars', 'allowed_keys', 'allowed_unicodes', 'clean_keys', 'clean_str', 'key_to_char', 'key_to_label', 'key_to_unicode', 'keys_to_str', 'label_to_char', 'label_to_key', 'label_to_unicode', 'labels_to_str', 'null_class', 'num_classes', 'str_to_keys', 'str_to_labels', 'unicode_to_char', 'unicode_to_key', 'unicode_to_label']


In [ ]:
import numpy as np
import torch
from emg2qwerty.charset import charset

def edit_distance_counts(ref: list, hyp: list) -> tuple[int, int, int]:
    R, H = len(ref), len(hyp)
    dp = np.zeros((R + 1, H + 1), dtype=int)
    dp[:, 0] = np.arange(R + 1)
    dp[0, :] = np.arange(H + 1)

    for i in range(1, R + 1):
        for j in range(1, H + 1):
            if ref[i - 1] == hyp[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j - 1],
                    dp[i - 1][j],
                    dp[i][j - 1],
                )

    # Backtrack to count S / D / I
    S = D = I = 0
    i, j = R, H
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            S += 1; i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1                             # insertion

    return S, D, I

def greedy_ctc_decode(emissions_np: np.ndarray, null_class: int) -> list[int]:
    seq = emissions_np.tolist()
    out, prev = [], None
    for c in seq:
        if c != null_class and c != prev:
            out.append(c)
        prev = c
    return out

def evaluate_cer(model, loader, device):
    model.eval()
    null_class = charset().null_class

    total_S = total_D = total_I = total_N = 0
    results = []

    with torch.no_grad():
        for batch in loader:
            inputs         = batch['inputs'].to(device)
            targets        = batch['targets']
            target_lengths = batch['target_lengths']

            emissions = model(inputs)
            greedy_batch = emissions.argmax(dim=-1).cpu().numpy()

            targets_np        = targets.numpy()
            target_lengths_np = target_lengths.numpy()

            for i in range(inputs.shape[1]):
                hyp = greedy_ctc_decode(greedy_batch[:, i], null_class)
                ref = targets_np[: target_lengths_np[i], i].tolist()

                S, D, I = edit_distance_counts(ref, hyp)
                N = len(ref)

                total_S += S; total_D += D; total_I += I; total_N += N

                ref_str = ''.join(charset().label_to_char(c) for c in ref)
                hyp_str = ''.join(charset().label_to_char(c) for c in hyp)
                cer = (S + D + I) / max(N, 1)
                results.append((ref_str, hyp_str, S, D, I, N, cer))

    overall_cer = (total_S + total_D + total_I) / max(total_N, 1)
    return overall_cer, total_S, total_D, total_I, total_N, results

device = next(model.parameters()).device
overall_cer, S, D, I, N, results = evaluate_cer(model, test_loader, device)

print('-' * 55)
print('  CER Evaluation Results')
print(f'  Substitutions (S) : {S:>8,}')
print(f'  Deletions     (D) : {D:>8,}')
print(f'  Insertions    (I) : {I:>8,}')
print(f'  Total errors  S+D+I : {S+D+I:>6,}')
print(f'  Ground-truth chars N: {N:>6,}')
print(f'  CER = (S+D+I)/N = {S+D+I}/{N} = {overall_cer:.4f} ({overall_cer*100:.2f}%)')

-------------------------------------------------------
  CER Evaluation Results
  Substitutions (S) :    2,657
  Deletions     (D) :      822
  Insertions    (I) :      167
  Total errors  S+D+I :  3,646
  Ground-truth chars N: 13,253
  CER = (S+D+I)/N = 3646/13253 = 0.2751 (27.51%)
